自定义层

1.不带参数的层

构造一个没有任何参数的自定义层,下面的CenteredLayer类要从其输入中减去均值。 要构建它，我们只需继承基础层类并实现前向传播功能。

In [1]:
import torch
import torch.nn.functional as F
from torch import nn


class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [2]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

将层作为组件合并到更复杂的模型中

In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

In [4]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(4.6566e-09, grad_fn=<MeanBackward0>)

2.带参数的层

需要两个参数，一个用于表示权重，另一个用于表示偏置项。  该层需要输入参数：in_units和units，分别表示输入数和输出数。

In [6]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

实例化并访问其中参数

In [7]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[ 0.2030, -0.7469,  0.1962],
        [-1.5059,  0.3733, -1.3958],
        [-0.0329, -1.1363, -1.6090],
        [ 0.0067, -0.5471, -0.5845],
        [ 0.7871, -2.0031, -0.2902]], requires_grad=True)

使用自定义曾直接进行前向传播计算，获得输出

In [8]:
linear(torch.rand(2, 5))

tensor([[0.0000, 0.0000, 0.0000],
        [0.0438, 0.0000, 0.0000]])

使用自定义层构建模型

In [9]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]])

## 练习

1. 设计一个接受输入并计算张量降维的层，它返回$y_k = \sum_{i, j} W_{ijk} x_i x_j$。


In [10]:
import torch
from torch import nn

class QuadraticLayer(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # W_{ijk}
        self.weight = nn.Parameter(
            torch.randn(in_features, in_features, out_features)
        )
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.bias = None

    def forward(self, x):
        # x shape: (batch_size, in_features)
        # 输出: y_k = sum_{i,j} W_{ijk} x_i x_j
        
        # einsum解释：
        # bi,bj,ijk -> bk
        # b: batch
        # i,j: 输入维度
        # k: 输出维度
        y = torch.einsum('bi,bj,ijk->bk', x, x, self.weight)
        
        if self.bias is not None:
            y = y + self.bias
        
        return y

In [11]:
X = torch.randn(4, 5)   # batch_size=4, 输入维度=5
layer = QuadraticLayer(in_features=5, out_features=3)
Y = layer(X)

print("X.shape =", X.shape)
print("Y.shape =", Y.shape)

X.shape = torch.Size([4, 5])
Y.shape = torch.Size([4, 3])


1. 设计一个返回输入数据的傅立叶系数前半部分的层。

In [12]:
import torch
from torch import nn

class FourierHalfLayer(nn.Module):
    def forward(self, x):
        # x shape: (batch_size, n)
        
        # 对最后一个维度做傅立叶变换
        X_fft = torch.fft.fft(x, dim=-1)
        
        n = x.shape[-1]
        
        # 返回前半部分频率系数
        return X_fft[..., :n // 2]

In [13]:
X = torch.randn(4, 8)   # batch_size=4, 每个样本长度=8
layer = FourierHalfLayer()
Y = layer(X)

print("X.shape =", X.shape)
print("Y.shape =", Y.shape)
print(Y)

X.shape = torch.Size([4, 8])
Y.shape = torch.Size([4, 4])
tensor([[-2.4794+0.0000j, -0.4591-3.9731j, -3.8222-2.2879j,  1.6202-0.0498j],
        [-2.6810+0.0000j,  4.8718+2.3681j, -0.0762+0.7354j, -3.9889-2.0500j],
        [-1.4803+0.0000j,  3.3236+0.2153j, -0.6780+1.3726j,  1.7669+3.1419j],
        [-1.4451+0.0000j, -0.2284+2.4677j,  2.6130-2.1184j,  2.4588+2.3422j]])
